# Sokoban

En aquesta pràctica desenvoluparem un agent capaç de resoldre Sokoban amb cerca en espais d'estats. L'estat només descriu el tauler; la cerca guarda per separat els costos i els pares.

![Sokoban](sokoban.png)

Un robot es mou en les quatre direccions i pot empènyer una caixa, però no arrossegar-la. L'objectiu és deixar totes les caixes en una posició d'emmagatzematge. Pots provar el joc a [Sokoban Online](http://www.sokobanonline.com/play).

## Exercici 1: l'estat

Representa l'estat amb una tupla `(robots, caixes)`. `robots` és una tupla de posicions i `caixes` és un `frozenset` de posicions.

Les dimensions, els magatzems i els obstacles no canvien dins d'un tauler: guarda'ls en un diccionari `problema`, separat de l'estat. No guardes dins l'estat ni l'acció, ni el cost, ni el pare: són dades de l'algorisme de cerca.

Treballa amb les llistes `exemples_facils`, `exemples_mitjans` i `exemples_dificils`. Cada element és una tupla `(nom, problema, inicial)`. Encara no cal resoldre'ls.

In [ ]:
Posicio = tuple[int, int]

def crea_problema(amplada, altura, magatzems, obstacles=()):
    return {
        'amplada': amplada,
        'altura': altura,
        'magatzems': frozenset(magatzems),
        'obstacles': frozenset(obstacles),
    }

exemples_facils = [
    ('Fàcil 1 — desviament voraç', crea_problema(5, 5, {(1, 4)}, {(1, 0), (0, 2), (1, 2)}), (((2, 4),), frozenset({(1, 3)}))),
    ('Fàcil 2 — empenta horitzontal', crea_problema(5, 5, {(4, 2)}), (((1, 2),), frozenset({(2, 2)}))),
    ('Fàcil 3 — empenta horitzontal inversa', crea_problema(5, 5, {(0, 3)}), (((3, 3),), frozenset({(2, 3)}))),
    ('Fàcil 4 — dos robots amb obstacles', crea_problema(5, 5, {(0, 0)}, {(1, 0), (2, 0), (3, 0), (1, 4), (2, 4), (3, 4)}), (((2, 1), (2, 3)), frozenset({(1, 1)}))),
]

exemples_mitjans = [
    ('Mitjà 1 — desviament voraç amb dues caixes', crea_problema(6, 4, {(2, 1), (2, 2)}), (((2, 1), (2, 2)), frozenset({(1, 1), (4, 2)}))),
    ('Mitjà 2 — obstacle central', crea_problema(6, 5, {(4, 1), (4, 3)}, {(3, 2)}), (((1, 1), (1, 3)), frozenset({(2, 1), (2, 3)}))),
    ('Mitjà 3 — caixes en paral·lel', crea_problema(7, 5, {(1, 0), (5, 4)}), (((1, 3), (5, 1)), frozenset({(1, 2), (5, 2)}))),
]

exemples_dificils = [
    ('Difícil 1 — zona oberta i desempat', crea_problema(8, 8, {(3, 1), (4, 6), (7, 0)}, {(0, 1), (1, 5), (2, 3), (3, 3), (5, 0)}), (((0, 6),), frozenset({(6, 3), (2, 5), (3, 6)}))),
    ('Difícil 2 — tres caixes verticals', crea_problema(10, 9, {(9, 0), (9, 4), (9, 8)}), (((0, 0), (0, 4), (0, 8)), frozenset({(1, 0), (1, 4), (1, 8)}))),
    ('Difícil 3 — tres caixes en tauler ample', crea_problema(12, 10, {(11, 1), (11, 4), (11, 7)}), (((0, 1), (0, 4), (0, 7)), frozenset({(1, 1), (1, 4), (1, 7)}))),
    ('Difícil 4 — tres carrils de 6×6', crea_problema(6, 6, {(5, 0), (5, 2), (0, 5)}), (((0, 0), (0, 2), (0, 4)), frozenset({(1, 0), (1, 2), (1, 4)}))),
    ('Difícil 5 — tres carrils de 7×7', crea_problema(7, 7, {(6, 0), (6, 3), (6, 6)}), (((0, 0), (0, 3), (0, 6)), frozenset({(1, 0), (1, 3), (1, 6)}))),
    ('Difícil 6 — tres carrils de 8×7', crea_problema(8, 7, {(7, 1), (7, 3), (7, 5)}), (((0, 1), (0, 3), (0, 5)), frozenset({(1, 1), (1, 3), (1, 5)}))),
    ('Difícil 7 — tres objectius amb recorregut lateral', crea_problema(8, 8, {(7, 0), (7, 3), (0, 7)}), (((0, 0), (0, 3), (0, 6)), frozenset({(1, 0), (1, 3), (1, 6)}))),
]

problema_facil, inicial_facil = exemples_facils[0][1:]
problema_mitja, inicial_mitja = exemples_mitjans[0][1:]
problema_dificil, inicial_dificil = exemples_dificils[0][1:]


## Exercici 2: successors

Implementa funcions, sense classes de problema:

- `dins_tauler(posicio, problema)`
- `successors(estat, problema)`, que retorne parells `(nou_estat, cost)`
- `es_objectiu(estat, problema)`
- `heuristica(estat, problema)`, amb la distància Manhattan de cada caixa al magatzem més proper.
- `mostra_estat(estat, problema)`, que represente el tauler amb emojis i un marc.

Tots els moviments tenen cost 1. Prova `successors(inicial_facil, problema_facil)` i comprova que cada moviment siga legal: no ix del tauler, no travessa obstacles i només empeny una caixa quan té espai al davant. Ordena sempre la tupla de robots abans de retornar un nou estat, per no representar el mateix tauler de dues maneres diferents.

In [ ]:
DIRECCIONS = ((0, 1), (0, -1), (1, 0), (-1, 0))


def dins_tauler(posicio, problema):
    x, y = posicio
    return 0 <= x < problema["amplada"] and 0 <= y < problema["altura"]


def successors(estat, problema):
    robots, caixes_actuals = estat
    resultats = []
    for robot in robots:
        for dx, dy in DIRECCIONS:
            desti = robot[0] + dx, robot[1] + dy
            if not dins_tauler(desti, problema) or desti in problema["obstacles"] or desti in robots:
                continue

            caixes = caixes_actuals
            if desti in caixes:
                desti_caixa = desti[0] + dx, desti[1] + dy
                bloquejat = problema["obstacles"] | caixes | frozenset(robots)
                if not dins_tauler(desti_caixa, problema) or desti_caixa in bloquejat:
                    continue
                caixes = frozenset((caixes - {desti}) | {desti_caixa})

            nous_robots = tuple(sorted((set(robots) - {robot}) | {desti}))
            resultats.append(((nous_robots, caixes), 1))
    return resultats


def es_objectiu(estat, problema):
    _, caixes = estat
    return caixes <= problema["magatzems"]


def heuristica(estat, problema):
    _, caixes = estat
    return sum(
        min(abs(caixa[0] - magatzem[0]) + abs(caixa[1] - magatzem[1]) for magatzem in problema["magatzems"])
        for caixa in caixes
    )


def reconstrueix_ruta(pares, final):
    ruta = [final]
    while pares[ruta[-1]] is not None:
        ruta.append(pares[ruta[-1]])
    return list(reversed(ruta))


def mostra_estat(estat, problema):
    robots, caixes = estat
    files = ["🚧" * (problema["amplada"] + 2)]
    for y in range(problema["altura"]):
        fila = ["🚧"]
        for x in range(problema["amplada"]):
            posicio = x, y
            if posicio in problema["obstacles"]:
                fila.append("🚧")
            elif posicio in robots:
                fila.append("🤖")
            elif posicio in caixes:
                fila.append("📦")
            elif posicio in problema["magatzems"]:
                fila.append("❌")
            else:
                fila.append("⬜")
        files.append("".join(fila + ["🚧"]))
    files.append("🚧" * (problema["amplada"] + 2))
    return "\n".join(files)


In [ ]:
print("Estat inicial:")
print(mostra_estat(inicial_facil, problema_facil))
print("Successors legals:")
for numero, (successor, cost) in enumerate(successors(inicial_facil, problema_facil), start=1):
    print(f"\nMoviment {numero} (cost {cost})")
    print(mostra_estat(successor, problema_facil))


## Exercici 3: UCS

Implementa `ucs(problema, inicial)`. La frontera és un `heapq`; `visitats` evita expandir un estat més d'una vegada i `pares` permet reconstruir la ruta.

Retorna `final, cost, ruta, visitats`, o `None, None, None, visitats` si no hi ha solució. `ruta` és la llista d'estats des de l'inicial fins al final; `visitats` compta els estats extrets de la frontera.

Resol `problema_facil` amb UCS i mostra el cost, els estats visitats i cada estat de la ruta.

In [ ]:
import heapq
from itertools import count


def ucs(problema, inicial, limit=100_000):
    ordre = count()
    frontera = [(0, next(ordre), inicial)]
    millor_cost = {inicial: 0}
    pares = {inicial: None}
    expandits = 0

    while frontera and expandits < limit:
        cost, _, estat = heapq.heappop(frontera)
        if cost != millor_cost[estat]:
            continue
        expandits += 1
        if es_objectiu(estat, problema):
            return estat, cost, reconstrueix_ruta(pares, estat), expandits

        for successor, cost_pas in successors(estat, problema):
            nou_cost = cost + cost_pas
            if nou_cost < millor_cost.get(successor, float("inf")):
                millor_cost[successor] = nou_cost
                pares[successor] = estat
                heapq.heappush(frontera, (nou_cost, next(ordre), successor))
    return None, None, None, expandits


In [ ]:
final, cost, ruta, expandits = ucs(problema_facil, inicial_facil)
print(f"Cost: {cost}; estats expandits: {expandits}")
for pas, estat in enumerate(ruta):
    print(f"\nPas {pas}")
    print(mostra_estat(estat, problema_facil))


## Exercici 4: Greedy i A*

Una heurística és **admissible** si mai no sobreestima el cost que falta fins a l'objectiu. La suma de les distàncies Manhattan de cada caixa al magatzem més proper és admissible, perquè ignora obstacles i interferències.

Implementa `greedy(problema, inicial)` amb prioritat `heuristica(estat)` i `a_star(problema, inicial)` amb prioritat `cost + heuristica(estat)`.

En A* usa el mateix esquema que en UCS. Compara el cost de les rutes i els estats visitats.

Resol els exemples fàcils amb Greedy i amb A*. Després prova els exemples mitjans: encara són prou petits per comparar els dos algorismes a aquest punt.

In [ ]:
def greedy(problema, inicial, limit=100_000):
    ordre = count()
    frontera = [(heuristica(inicial, problema), next(ordre), 0, inicial)]
    visitats = set()
    pares = {inicial: None}

    while frontera and len(visitats) < limit:
        _, _, cost, estat = heapq.heappop(frontera)
        if estat in visitats:
            continue
        visitats.add(estat)
        if es_objectiu(estat, problema):
            return estat, cost, reconstrueix_ruta(pares, estat), len(visitats)

        for successor, cost_pas in successors(estat, problema):
            if successor not in visitats and successor not in pares:
                pares[successor] = estat
                heapq.heappush(
                    frontera,
                    (heuristica(successor, problema), next(ordre), cost + cost_pas, successor),
                )
    return None, None, None, len(visitats)


def a_star(problema, inicial, limit=100_000):
    ordre = count()
    frontera = [(heuristica(inicial, problema), 0, next(ordre), inicial)]
    millor_cost = {inicial: 0}
    pares = {inicial: None}
    expandits = 0

    while frontera and expandits < limit:
        _, cost, _, estat = heapq.heappop(frontera)
        if cost != millor_cost[estat]:
            continue
        expandits += 1
        if es_objectiu(estat, problema):
            return estat, cost, reconstrueix_ruta(pares, estat), expandits

        for successor, cost_pas in successors(estat, problema):
            nou_cost = cost + cost_pas
            if nou_cost < millor_cost.get(successor, float("inf")):
                millor_cost[successor] = nou_cost
                pares[successor] = estat
                prioritat = nou_cost + heuristica(successor, problema)
                heapq.heappush(frontera, (prioritat, nou_cost, next(ordre), successor))
    return None, None, None, expandits


## Exercici 5: comparació

Ja tens quatre taulers fàcils, tres mitjans i set difícils. El primer de cada categoria està dissenyat perquè Greedy trobe una ruta no òptima. Compara UCS, Greedy i A*: cost, longitud de la ruta, estats visitats i temps d'execució. Pots utilitzar `time.time()` per mesurar el temps.

Quin algorisme és més eficient en temps i estats visitats? Quin troba sempre la ruta òptima? Per què? Quina informació aporta l'heurística?

Completa la comparació amb cada exemple difícil només amb Greedy i A*. UCS pot ser massa lent en aquests nivells. Usa un límit d'estats expandits per evitar que una execució massa llarga bloquege el notebook; si s'arriba al límit, informa-ho com a resultat.

In [ ]:
from time import perf_counter


def compara(nom, problema, inicial, algorismes):
    print(f"\n{nom}")
    for etiqueta, cerca in algorismes:
        inici = perf_counter()
        final, cost, ruta, expandits = cerca(problema, inicial)
        segons = perf_counter() - inici
        passos = None if ruta is None else len(ruta) - 1
        print(f"{etiqueta}: cost={cost}, passos={passos}, expandits={expandits}, temps={segons:.3f} s")


for exemple in exemples_facils + exemples_mitjans:
    compara(*exemple, [("UCS", ucs), ("Greedy", greedy), ("A*", a_star)])


## Exercici 6: desempats en A*

Els taulers difícils poden ser lents: en una zona oberta, molts estats tenen la mateixa prioritat `f = g + h`, perquè l'heurística és poc informativa. A* ha d'escollir entre molts candidats igualment bons.

Prova un **tie breaking**: quan dos estats tenen el mateix valor `f`, prioritza el que té un `g` més gran. Amb `heapq`, pots usar una prioritat `(f, -g, comptador, estat)`. El comptador només evita que Python intente comparar directament dos estats.

Implementa `a_star_millorat(problema, inicial)` i compara'l amb A* en els taulers difícils. El desempat no canvia l'optimalitat; només l'ordre d'exploració.

Resol els exemples difícils amb les dues versions d'A* i compara els estats visitats. En el primer exemple difícil, A* sense desempat pot arribar al límit abans de trobar la solució; això forma part de la comparació.

In [ ]:
def a_star_millorat(problema, inicial, limit=100_000):
    ordre = count()
    frontera = [(heuristica(inicial, problema), 0, next(ordre), inicial)]
    millor_cost = {inicial: 0}
    pares = {inicial: None}
    expandits = 0

    while frontera and expandits < limit:
        _, menys_cost, _, estat = heapq.heappop(frontera)
        cost = -menys_cost
        if cost != millor_cost[estat]:
            continue
        expandits += 1
        if es_objectiu(estat, problema):
            return estat, cost, reconstrueix_ruta(pares, estat), expandits

        for successor, cost_pas in successors(estat, problema):
            nou_cost = cost + cost_pas
            if nou_cost < millor_cost.get(successor, float("inf")):
                millor_cost[successor] = nou_cost
                pares[successor] = estat
                prioritat = nou_cost + heuristica(successor, problema)
                heapq.heappush(frontera, (prioritat, -nou_cost, next(ordre), successor))
    return None, None, None, expandits


for nom, problema, inicial in exemples_dificils:
    compara(nom, problema, inicial, [("Greedy", greedy), ("A*", a_star), ("A* millorat", a_star_millorat)])
